%md
# Data Management 2 - Final Project: Sales Data Pipeline

**Goal:** Combine two different data sources through a Medallion (Bronze → Silver → Gold) pipeline on Databricks and report two business objectives in a dashboard.

## Data sources (two different types)
- **Products** : a Delta **table** (database source): StockCode, Description, UnitPrice.
- **Orders** : **streaming JSON files** read with Auto Loader (stream source): InvoiceNo, StockCode, Quantity, Country, InvoiceDate.
- **Join key:** `StockCode`

## Business objectives
1. **Revenue by region** : total revenue per country.
2. **Top 10 products by units sold.**

## Architecture
Bronze (raw) → Silver (cleaned + joined) → Gold (aggregated per objective) → AI/BI Dashboard.

## Data source (cited)
Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33 (CC BY 4.0). Both sources are derived from this single cited dataset, as permitted by the project rules.

%md
## Step 1 - Load raw data
Read the Online Retail CSV from the Unity Catalog Volume to confirm it loaded.           

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/dm2_raw/Online Retail.csv",
    header=True, inferSchema=True
)
print(df.count(), "rows")
display(df.limit(10))

541909 rows


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01/12/10 08:26,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,01/12/10 08:26,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01/12/10 08:26,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01/12/10 08:26,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01/12/10 08:26,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,01/12/10 08:26,7.65,17850,United Kingdom
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,01/12/10 08:26,4.25,17850,United Kingdom
536366,22633,HAND WARMER UNION JACK,6,01/12/10 08:28,1.85,17850,United Kingdom
536366,22632,HAND WARMER RED POLKA DOT,6,01/12/10 08:28,1.85,17850,United Kingdom
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,01/12/10 08:34,1.69,13047,United Kingdom


%md
## Step 2 — Build the two source systems
Light cleaning (remove cancellations, nulls, non-positive values; keep a small subset of countries), then split the dataset into the two sources: the `products` Delta table (DB source) and the `orders_stream/` JSON files (stream source).

In [0]:
from pyspark.sql import functions as F

raw = spark.read.csv(
    "/Volumes/workspace/default/dm2_raw/Online Retail.csv",
    header=True, inferSchema=True
)

clean = (
    raw
    .filter(F.col("StockCode").isNotNull() & F.col("Description").isNotNull())
    .filter(~F.col("InvoiceNo").cast("string").startswith("C"))   # drop cancellations
    .filter((F.col("Quantity") > 0) & (F.col("UnitPrice") > 0))
)

# stay small (per "don't go big") — a few countries only
keep = ["United Kingdom", "Germany", "France", "Spain", "Netherlands"]
clean = clean.filter(F.col("Country").isin(keep))

print("rows after cleaning/subset:", clean.count())

rows after cleaning/subset: 507413


In [0]:
products = (
    clean.groupBy("StockCode")
         .agg(
             F.first("Description").alias("Description"),
             F.round(F.avg("UnitPrice"), 2).alias("UnitPrice")
         )
)

products.write.mode("overwrite").saveAsTable("workspace.default.products")
print("products rows:", spark.table("workspace.default.products").count())
display(spark.table("workspace.default.products").limit(5))

products rows: 3917


StockCode,Description,UnitPrice
21756,BATH BUILDING BLOCK WORD,6.63
84854,GIRLY PINK TOOL SET,4.95
15056N,EDWARDIAN PARASOL NATURAL,7.01
21340,CLASSIC METAL BIRDCAGE PLANT HOLDER,13.4
22451,SILK PURSE BABUSHKA RED,4.2


In [0]:
orders = clean.select("InvoiceNo", "StockCode", "Quantity", "Country", "InvoiceDate")

orders_path = "/Volumes/workspace/default/dm2_raw/orders_stream"

(orders
    .repartition(3)                       # ~3 files = simulated feed
    .write.mode("overwrite").format("json")
    .save(orders_path))

display(dbutils.fs.ls(orders_path))       # confirm files landed

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/_SUCCESS,_SUCCESS,0,1782042783000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/_committed_5396493436585520636,_committed_5396493436585520636,294,1782042782000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/_started_5396493436585520636,_started_5396493436585520636,0,1782042781000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/part-00000-tid-5396493436585520636-543ec065-cf25-4cde-8a78-05d7115193bd-232-1-c000.json,part-00000-tid-5396493436585520636-543ec065-cf25-4cde-8a78-05d7115193bd-232-1-c000.json,19293240,1782042781000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/part-00001-tid-5396493436585520636-543ec065-cf25-4cde-8a78-05d7115193bd-233-1-c000.json,part-00001-tid-5396493436585520636-543ec065-cf25-4cde-8a78-05d7115193bd-233-1-c000.json,19292894,1782042781000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/part-00002-tid-5396493436585520636-543ec065-cf25-4cde-8a78-05d7115193bd-234-1-c000.json,part-00002-tid-5396493436585520636-543ec065-cf25-4cde-8a78-05d7115193bd-234-1-c000.json,19292968,1782042781000


%md
## Step 3 — Bronze layer (raw ingestion)
Land both sources as-is into Delta tables with an ingestion timestamp.
- `bronze_products` : batch read from the products table.
- `bronze_orders` : **Auto Loader** stream from the orders folder (`availableNow` trigger, so re-running ingests only new files).

In [0]:
from pyspark.sql import functions as F

(spark.table("workspace.default.products")
    .withColumn("ingest_ts", F.current_timestamp())
    .write.mode("overwrite")
    .saveAsTable("workspace.default.bronze_products"))

print("bronze_products rows:", spark.table("workspace.default.bronze_products").count())

bronze_products rows: 3917


In [0]:
orders_path  = "/Volumes/workspace/default/dm2_raw/orders_stream"
schema_path  = "/Volumes/workspace/default/dm2_raw/_schema_orders"
chk_path     = "/Volumes/workspace/default/dm2_raw/_chk_orders"

stream = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(orders_path)
    .withColumn("ingest_ts", F.current_timestamp()))

q = (stream.writeStream
    .option("checkpointLocation", chk_path)
    .trigger(availableNow=True)
    .toTable("workspace.default.bronze_orders"))

q.awaitTermination()   # waits until all current files are processed, then stops
print("done ingesting")

done ingesting


In [0]:
print("bronze_orders rows:", spark.table("workspace.default.bronze_orders").count())
display(spark.table("workspace.default.bronze_orders").limit(10))

bronze_orders rows: 507413


Country,InvoiceDate,InvoiceNo,Quantity,StockCode,_rescued_data,ingest_ts
United Kingdom,01/12/10 08:26,536365,6,21730,null,2026-06-21T11:56:50.201Z
United Kingdom,01/12/10 08:26,536365,6,84029E,null,2026-06-21T11:56:50.201Z
United Kingdom,01/12/10 08:26,536365,6,85123A,null,2026-06-21T11:56:50.201Z
United Kingdom,01/12/10 08:34,536367,3,21754,null,2026-06-21T11:56:50.201Z
United Kingdom,01/12/10 08:34,536367,6,22310,null,2026-06-21T11:56:50.201Z
United Kingdom,01/12/10 08:34,536367,6,22745,null,2026-06-21T11:56:50.201Z
United Kingdom,01/12/10 08:34,536367,4,48187,null,2026-06-21T11:56:50.201Z
United Kingdom,01/12/10 08:34,536368,3,22912,null,2026-06-21T11:56:50.201Z
United Kingdom,01/12/10 08:34,536368,6,22960,null,2026-06-21T11:56:50.201Z
France,01/12/10 08:45,536370,18,21035,null,2026-06-21T11:56:50.201Z


%md
## Step 4 — Silver layer (clean + join)
Cast types, drop nulls/duplicates, and **join the two sources** on `StockCode`, computing `revenue = Quantity × UnitPrice`. Output: `silver_orders_enriched`.

In [0]:
from pyspark.sql import functions as F

orders_clean = (spark.table("workspace.default.bronze_orders")
    .withColumn("StockCode", F.trim(F.col("StockCode").cast("string")))
    .withColumn("Quantity", F.col("Quantity").cast("int"))
    .filter(F.col("StockCode").isNotNull() & F.col("Quantity").isNotNull())
    .dropDuplicates(["InvoiceNo", "StockCode", "Quantity"]))

products_clean = (spark.table("workspace.default.bronze_products")
    .withColumn("StockCode", F.trim(F.col("StockCode").cast("string")))
    .withColumn("UnitPrice", F.col("UnitPrice").cast("double"))
    .select("StockCode", "Description", "UnitPrice"))

In [0]:
silver = (orders_clean.join(products_clean, on="StockCode", how="inner")
    .withColumn("revenue", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
    .select(
        "InvoiceNo", "StockCode", "Description",
        "Quantity", "UnitPrice", "revenue",
        F.col("Country").alias("Region"),
        "InvoiceDate"
    ))

silver.write.mode("overwrite").saveAsTable("workspace.default.silver_orders_enriched")

print("silver rows:", spark.table("workspace.default.silver_orders_enriched").count())
display(spark.table("workspace.default.silver_orders_enriched").limit(10))

silver rows: 502163


InvoiceNo,StockCode,Description,Quantity,UnitPrice,revenue,Region,InvoiceDate
536373,82482,WOODEN PICTURE FRAME WHITE FINISH,6,3.11,18.66,United Kingdom,01/12/10 09:02
536373,84406B,CREAM CUPID HEARTS COAT HANGER,8,4.32,34.56,United Kingdom,01/12/10 09:02
536375,20679,EDWARDIAN PARASOL RED,6,7.09,42.54,United Kingdom,01/12/10 09:32
536375,82483,WOOD 2 DRAWER CABINET WHITE FINISH,2,6.95,13.9,United Kingdom,01/12/10 09:32
536381,22083,PAPER CHAIN KIT RETROSPOT,1,3.79,3.79,United Kingdom,01/12/10 09:41
536388,22922,FRIDGE MAGNETS US DINER ASSORTED,12,1.89,22.68,United Kingdom,01/12/10 09:59
536396,21071,VINTAGE BILLBOARD DRINK ME MUG,6,1.71,10.26,United Kingdom,01/12/10 10:51
536398,22835,HOT WATER BOTTLE I AM SO POORLY,8,5.9,47.2,United Kingdom,01/12/10 10:52
536401,21463,MIRRORED DISCO BALL,1,5.8,5.8,United Kingdom,01/12/10 11:21
536401,21592,RETROSPOT CIGAR BOX MATCHES,1,1.13,1.13,United Kingdom,01/12/10 11:21


%md
## Step 5 - Gold layer (business objectives)
Two aggregated tables, one per objective:
- `gold_revenue_by_region` : total revenue per region.
- `gold_top_products` : units sold per product (dashboard shows top 10).

In [0]:
from pyspark.sql import functions as F

gold_region = (spark.table("workspace.default.silver_orders_enriched")
    .groupBy("Region")
    .agg(
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.countDistinct("InvoiceNo").alias("num_orders")
    )
    .orderBy(F.desc("total_revenue")))

gold_region.write.mode("overwrite").saveAsTable("workspace.default.gold_revenue_by_region")
display(spark.table("workspace.default.gold_revenue_by_region"))

Region,total_revenue,num_orders
United Kingdom,1.163348193E7,18019
Netherlands,405911.25,94
Germany,293069.91,457
France,253632.22,392
Spain,75863.9,90


In [0]:
gold_products = (spark.table("workspace.default.silver_orders_enriched")
    .groupBy("StockCode", "Description")
    .agg(
        F.sum("Quantity").alias("total_units"),
        F.round(F.sum("revenue"), 2).alias("total_revenue")
    )
    .orderBy(F.desc("total_units")))

gold_products.write.mode("overwrite").saveAsTable("workspace.default.gold_top_products")
display(spark.table("workspace.default.gold_top_products").limit(10))

StockCode,Description,total_units,total_revenue
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.6
23166,MEDIUM CERAMIC TOP STORAGE JAR,77156,114962.44
22197,SMALL POPCORN HOLDER,53995,56694.75
84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,50342,16109.44
85099B,JUMBO BAG RED RETROSPOT,47164,117910.0
85123A,WHITE HANGING HEART T-LIGHT HOLDER,36059,112864.67
84879,ASSORTED COLOUR BIRD ORNAMENT,35793,61563.96
21212,PACK OF 72 RETROSPOT CAKE CASES,31533,24280.41
22616,PACK OF 12 LONDON TISSUES,25675,11553.75
23084,RABBIT NIGHT LIGHT,24558,59184.78


%md
## Step 6 : Dashboard
Two AI/BI dashboard tiles built on the Gold tables: **Revenue by Region** and **Top 10 Products by Units Sold** (built in the Dashboards UI - see screenshots).

%md
## Step 7 — Incremental test (new data → dashboard updates)
Append new orders to the stream source, then re-run Bronze → Silver → Gold. After a dashboard refresh the numbers change (Germany revenue rose from ~293K to ~386K), demonstrating the pipeline reacts to new data as required.

In [0]:
from pyspark.sql import functions as F

new_orders = (spark.table("workspace.default.products")
    .select("StockCode").limit(3)
    .withColumn("InvoiceNo", F.lit("TEST001"))
    .withColumn("Quantity", F.lit(5000))
    .withColumn("Country", F.lit("Germany"))
    .withColumn("InvoiceDate", F.lit("12/9/2011 10:00"))
    .select("InvoiceNo","StockCode","Quantity","Country","InvoiceDate"))

(new_orders.repartition(1).write.mode("append").format("json")
    .save("/Volumes/workspace/default/dm2_raw/orders_stream"))

print("files now in the folder:")
display(dbutils.fs.ls("/Volumes/workspace/default/dm2_raw/orders_stream"))

files now in the folder:


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/_SUCCESS,_SUCCESS,0,1782058191000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/_committed_2105427434289958026,_committed_2105427434289958026,114,1782057460000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/_committed_3115872302787937764,_committed_3115872302787937764,114,1782058191000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/_committed_5396493436585520636,_committed_5396493436585520636,294,1782042782000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/_committed_vacuum1426855644114672446,_committed_vacuum1426855644114672446,96,1782057461000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/_started_2105427434289958026,_started_2105427434289958026,0,1782057460000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/_started_3115872302787937764,_started_3115872302787937764,0,1782058191000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/part-00000-tid-2105427434289958026-2778300c-9ac9-4832-8a73-4bd6ee82393c-130-1-c000.json,part-00000-tid-2105427434289958026-2778300c-9ac9-4832-8a73-4bd6ee82393c-130-1-c000.json,337,1782057460000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/part-00000-tid-3115872302787937764-f7d90a01-226a-4df6-8f06-ac0dafcac806-137-1-c000.json,part-00000-tid-3115872302787937764-f7d90a01-226a-4df6-8f06-ac0dafcac806-137-1-c000.json,337,1782058191000
dbfs:/Volumes/workspace/default/dm2_raw/orders_stream/part-00000-tid-5396493436585520636-543ec065-cf25-4cde-8a78-05d7115193bd-232-1-c000.json,part-00000-tid-5396493436585520636-543ec065-cf25-4cde-8a78-05d7115193bd-232-1-c000.json,19293240,1782042781000


In [0]:
stream = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/dm2_raw/_schema_orders")
    .load("/Volumes/workspace/default/dm2_raw/orders_stream")
    .withColumn("ingest_ts", F.current_timestamp()))

q = (stream.writeStream
    .option("checkpointLocation", "/Volumes/workspace/default/dm2_raw/_chk_orders")
    .trigger(availableNow=True)
    .toTable("workspace.default.bronze_orders"))
q.awaitTermination()

print("Bronze TEST rows:", spark.table("workspace.default.bronze_orders").filter("InvoiceNo='TEST001'").count())

Bronze TEST rows: 6


In [0]:
orders_clean = (spark.table("workspace.default.bronze_orders")
    .withColumn("StockCode", F.trim(F.col("StockCode").cast("string")))
    .withColumn("Quantity", F.col("Quantity").cast("int"))
    .filter(F.col("StockCode").isNotNull() & F.col("Quantity").isNotNull())
    .dropDuplicates(["InvoiceNo","StockCode","Quantity"]))

products_clean = (spark.table("workspace.default.bronze_products")
    .withColumn("StockCode", F.trim(F.col("StockCode").cast("string")))
    .withColumn("UnitPrice", F.col("UnitPrice").cast("double"))
    .select("StockCode","Description","UnitPrice"))

(orders_clean.join(products_clean, on="StockCode", how="inner")
    .withColumn("revenue", F.round(F.col("Quantity")*F.col("UnitPrice"),2))
    .select("InvoiceNo","StockCode","Description","Quantity","UnitPrice","revenue",
            F.col("Country").alias("Region"),"InvoiceDate")
    .write.mode("overwrite").saveAsTable("workspace.default.silver_orders_enriched"))

(spark.table("workspace.default.silver_orders_enriched").groupBy("Region")
    .agg(F.round(F.sum("revenue"),2).alias("total_revenue"),
         F.countDistinct("InvoiceNo").alias("num_orders"))
    .orderBy(F.desc("total_revenue"))
    .write.mode("overwrite").saveAsTable("workspace.default.gold_revenue_by_region"))

(spark.table("workspace.default.silver_orders_enriched").groupBy("StockCode","Description")
    .agg(F.sum("Quantity").alias("total_units"),
         F.round(F.sum("revenue"),2).alias("total_revenue"))
    .orderBy(F.desc("total_units"))
    .write.mode("overwrite").saveAsTable("workspace.default.gold_top_products"))

display(spark.table("workspace.default.gold_revenue_by_region"))

Region,total_revenue,num_orders
United Kingdom,1.163348193E7,18019
Netherlands,405911.25,94
Germany,386019.91,458
France,253632.22,392
Spain,75863.9,90
